In [ ]:
# thư viện và dữ liệu
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')
from prophet import Prophet
from prophet.diagnostics import cross_validation, performance_metrics
from prophet.plot import plot_cross_validation_metric

# tải dữ liệu (đường dẫn từ thư mục Forecasting Product)
df        = pd.read_csv('../tnbike_data.csv', parse_dates=['ds'])
future_df = pd.read_csv('../future.csv',      parse_dates=['ds'])
df

In [ ]:
# gộp 2 dataset
df = pd.concat([df, future_df]).reset_index(drop=True)
df.tail()

In [ ]:
# đổi tên biến
df = df.rename(columns={'revenue': 'y'})
df.head(1)

In [ ]:
# biến ngày
df['ds'] = pd.to_datetime(df['ds'])
df.ds

# Ngày lễ

In [ ]:
# Tết Nguyên Đán
dates_tet = pd.to_datetime(df[df.is_tet == 1].ds)
tet = pd.DataFrame({"holiday": "tet_nguyen_dan", "ds": dates_tet,
                    "lower_window": -7, "upper_window": 7})

In [ ]:
# Ngày Thiếu nhi 1/6
dates_thieu_nhi = pd.to_datetime(df[df.is_thieu_nhi == 1].ds)
thieu_nhi = pd.DataFrame({"holiday": "ngay_thieu_nhi", "ds": dates_thieu_nhi,
                           "lower_window": -14, "upper_window": 3})

In [ ]:
thieu_nhi

In [ ]:
# gộp ngày lễ
holidays = pd.concat([tet, thieu_nhi])
holidays

In [ ]:
df = df.drop(['is_tet', 'is_thieu_nhi'], axis=1)

In [ ]:
# tách tập train và tập dự báo
training  = df.iloc[:-3, :]
future_df = df.iloc[-3:, :]

In [ ]:
# lấy tham số tốt nhất
parameters = pd.read_csv('best_params_prophet.csv', index_col=0)

In [ ]:
# trích xuất giá trị tham số
changepoint_prior_scale = float(parameters.loc['changepoint_prior_scale'][0])
holidays_prior_scale    = float(parameters.loc['holidays_prior_scale'][0])
seasonality_prior_scale = float(parameters.loc['seasonality_prior_scale'][0])
seasonality_mode        = parameters.loc['seasonality_mode'][0]

In [ ]:
# mô hình Prophet
m = Prophet(holidays=holidays,
            seasonality_mode=seasonality_mode,
            seasonality_prior_scale=seasonality_prior_scale,
            holidays_prior_scale=holidays_prior_scale,
            changepoint_prior_scale=changepoint_prior_scale)
m.add_seasonality(name='monthly', period=30.5, fourier_order=5)
m.fit(training)

# Dự báo

In [ ]:
# tạo dataframe tương lai
future = m.make_future_dataframe(periods=3, freq='MS')
future.tail()

In [ ]:
# dự báo
forecast = m.predict(future)
forecast[['ds','yhat','yhat_lower','yhat_upper']].tail(3)

In [ ]:
m.plot_components(forecast);

In [ ]:
# trích xuất kết quả dự báo Q2/2026
predictions_prophet = forecast[['ds','yhat','yhat_lower','yhat_upper']].tail(3)
predictions_prophet.index = future_df['ds'].values

In [ ]:
# xuất kết quả
predictions_prophet.to_csv('Ensemble/predictions_prophet.csv', index=False)
print('✅ Đã lưu Ensemble/predictions_prophet.csv')